
#Ingeset results.json file

1. Read the file using spark dataframe reader API
2. Define and enforce schena (preserve the nested structure)
3. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
4. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

### Step 1 - Read the JSON file using DataFrameReader API

In [0]:
results_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType())
])

In [0]:
results_df = (
    spark.read.format('json')
    .schema(results_schema)
    .option('mode','FAILFAST')
    .load(source_file)
) 


### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
results_final_df = add_ingestion_metadata(results_df)


### Step 3 - Write to Bronze Delta Table

In [0]:
write_to_bronze(input_df = results_final_df, table_name = table_name, batch_id = v_batch_id)

In [0]:
# display(spark.table(table_name))

In [0]:
# %sql
# SELECT season, COUNT(*) FROM formula1.bronze.results GROUP BY season ORDER BY season;